# 02 - Dataset Harmonization and Master Manifest Builder
### UROP: Improving Cross-Dataset Generalization and Severe-Stage Detection in ROP
This notebook standardizes all heterogeneous dataset labels into a unified clinical schema:
- Unique patient identifiers with dataset prefixes (`ROPVL_0001`, `OST_043`, `FARFUM_012`).
- Preserves original raw labels and maps harmonized binary / severity labels.
- Computes **Stratified Group 5-Fold Cross-Validation** partitioned strictly at the patient level.
- Executes automated assertions ensuring **0% patient data leakage** across splits.


In [1]:
import os, sys
if "COLAB_GPU" in os.environ or os.path.exists("/content"):
    sys.path.append("/content/drive/MyDrive/ROP_UROP")
    sys.path.append("/content/ROP_UROP")
else:
    sys.path.append(r"d:/UROP")

from src.manifest import build_manifests
build_manifests()


BUILDING MASTER MANIFEST AND PATIENT-LEVEL FOLDS
Loaded 10841 source rows from D:\UROP\manifest.csv

Executing automated data-leakage assertions...
[PASS] Automated assertions verified: 0 patient overlap across all folds.

[OK] Master manifest saved to D:\UROP\manifests\master_manifest.csv (10841 rows)
[OK] Task A manifest saved to D:\UROP\manifests\taskA_manifest.csv (6145 images: 3703 Normal, 2442 ROP)
[OK] Task B manifest saved to D:\UROP\manifests\taskB_manifest.csv (1052 images: mild/moderate/severe)

Task A Fold Distribution (Images per Fold):
col_0       HVDROPDB_Class0  HVDROPDB_Class1  OSTRAVA_Class0  OSTRAVA_Class1  ROPVL_Class0  ROPVL_Class1
fold_taskA                                                                                              
-1                      100               85               0               0             0             0
 0                        0                0             727             505           115           212
 1                      

In [2]:
import pandas as pd
from IPython.display import display

manifest_path = r"d:/UROP\manifests\master_manifest.csv" if os.path.exists(r"d:/UROP") else "manifests/master_manifest.csv"
master_df = pd.read_csv(manifest_path)
print("Total rows in Master Manifest:", len(master_df))
display(master_df.head(5))


Total rows in Master Manifest: 10841


,image_path,dataset_source,country,hospital,camera_device,original_patient_id,unique_patient_id,exam_id,eye,visit_date,...,binary_label,severity_label,image_width,image_height,ga,bw,exclude_reason,split,fold_taskA,fold_taskB
0,D:\UROP\ROP-VL\Image\Normal\1_2020-04-03_L_3.jpg,ROPVL,China,Guangdong Women & Children Hospital,Camera_G3_>1200,1,ROPVL_1,ROPVL_1_2020-04-03,L,2020-04-03,...,0.0,NaN,1454,1454,34.0,2650.0,NaN,train_val,3,-1
1,D:\UROP\ROP-VL\Image\Normal\1_2020-04-03_R_11.jpg,ROPVL,China,Guangdong Women & Children Hospital,Camera_G3_>1200,1,ROPVL_1,ROPVL_1_2020-04-03,R,2020-04-03,...,0.0,NaN,1458,1458,34.0,2650.0,NaN,train_val,3,-1
2,D:\UROP\ROP-VL\Image\Normal\2_2008-10-15_L_24.jpg,ROPVL,China,Guangdong Women & Children Hospital,Camera_G1_<900,2,ROPVL_2,ROPVL_2_2008-10-15,L,2008-10-15,...,0.0,NaN,613,613,32.0,1600.0,NaN,train_val,0,-1
3,D:\UROP\ROP-VL\Image\Normal\2_2008-10-15_R_1.jpg,ROPVL,China,Guangdong Women & Children Hospital,Camera_G1_<900,2,ROPVL_2,ROPVL_2_2008-10-15,R,2008-10-15,...,0.0,NaN,608,608,32.0,1600.0,NaN,train_val,0,-1
4,D:\UROP\ROP-VL\Image\Normal\3_2022-04-21_L_1.jpg,ROPVL,China,Guangdong Women & Children Hospital,Camera_G3_>1200,3,ROPVL_3,ROPVL_3_2022-04-21,L,2022-04-21,...,0.0,NaN,1420,1420,37.0,2200.0,NaN,train_val,1,-1


In [4]:
# Verify Task A Screening Cohort & Patient Splits
taskA_path = r"d:/UROP\manifests\taskA_manifest.csv" if os.path.exists(r"d:/UROP") else "manifests/taskA_manifest.csv"
taskA_df = pd.read_csv(taskA_path)
print("Task A images per fold:")
display(pd.crosstab(taskA_df["fold_taskA"], taskA_df["dataset_source"] + "_Binary" + taskA_df["binary_label"].astype(int).astype(str)))


Task A images per fold:


col_0,HVDROPDB_Binary0,HVDROPDB_Binary1,OSTRAVA_Binary0,OSTRAVA_Binary1,ROPVL_Binary0,ROPVL_Binary1
fold_taskA,,,,,,
-1,100,85,0,0,0,0
0,0,0,727,505,115,212
1,0,0,624,395,133,199
2,0,0,518,97,123,221
3,0,0,578,156,125,197
4,0,0,533,152,127,223
